# Worked solutions

This notebook is identical to the student version except that every 🔵 `# TODO` has been filled in, **with commentary on why the answer is what it is** rather than just the code. Read the comments — the reasoning is the point, not the syntax.

Everything else, including the ✏️ YOUR TURN cells, is unchanged: those have no single right answer.


# Module A — Structural MRI

## Can a brain scan tell us who has dementia?

> ⚠️ **Real OASIS measurements; simulated slice images. Section 1.5 explains exactly what that means.**

### What you will be able to do by the end

1. read the numbers a radiologist's pipeline extracts from a brain scan, and say what each one means
2. explain why the same person appearing twice in a dataset destroys a test score, and fix it
3. train a **support vector machine** directly on brain images
4. train a small **convolutional neural network** on the same images, on your own laptop's CPU
5. explain why the CNN does not win, and why that is the expected answer at this sample size

### The data

**Real data, simulated pictures.** The table is **OASIS-2**: 373 real MRI sessions from 150 real older adults, with the volumetric measures a processing pipeline extracted from each scan (`eTIV`, `nWBV`, `ASF`) and real clinical ratings. Many participants were scanned several times, which is exactly what makes this module's central lesson possible.

The **2D slice images** are simulated. Raw OASIS images need a signed data-use agreement, so we cannot ship them. Instead each visit gets a 64×64 phantom slice whose ventricle size and cortical ribbon are drawn from *that visit's real measured brain volume*. The pixels are ours; the anatomy they encode is a real measurement of a real person.

### How to work through this notebook

Run the cells in order, top to bottom. The notebook is split into four sections:

| | Section | What happens |
|---|---|---|
| 1 | **Understand the data** | Meet every column and every person in the table |
| 2 | **Quality control** | Find the flaws before they fool you |
| 3 | **Build models** | Start from something trivial, then climb |
| 4 | **Read the results** | Turn numbers into a clinical judgement |

Look out for these markers:

- ✏️ **YOUR TURN** — change the value shown, re-run the cell, watch the figure change. Everyone does these.
- 🟢 run and read · 🔵 write a little code · ⚫ take home
- 🧠 a question to think about; the answer is hidden underneath, so try first

**In a hurry?** Read section 2's figures without doing the ✏️ turns, then go to 3.3 (the SVM on images) and work forward.

---

*Teaching material. Nothing here is a diagnostic tool, and no result in this notebook is clinical evidence.*


In [ ]:
# Run me first. This finds the project folder, loads the shared helpers,
# and prints exactly where this module's data came from.
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plots
from data import load_data, load_extra, provenance
from models import split_data, train_model, evaluate, compare_models, sweep_parameter, MODEL_CHOICES
import images as image_tools

pd.set_option('display.width', 160)
print(provenance('A'))


---
# 1 · Understand the data

One row is **one MRI session**, not one person. Hold on to that distinction — it is the whole of section 2.


### 1.1 Load the table


In [ ]:
df = load_data('A')
print(f'{df.shape[0]} scanning sessions from {df.subject_id.nunique()} different people.')
df.head(6)


### 1.2 What the columns mean

A structural MRI is a 3D picture of brain tissue. Nobody feeds the raw picture to a doctor; a processing pipeline first reduces it to a handful of numbers.

| Column | What it is |
|---|---|
| `subject_id` | The person. **Appears more than once.** |
| `session_id` | One scanning appointment. |
| `visit`, `days_since_first_visit` | Which appointment this was, and how long after the first. |
| `group` | `Nondemented`, `Demented`, or `Converted` — someone who was fine at first and was not later. |
| `age`, `sex` | At the time of this scan. |
| `education_years` | Years of schooling — a proxy for **cognitive reserve**. |
| `ses` | Socioeconomic status, 1 (highest) to 5 (lowest). Note how often it is missing. |
| `mmse` | Mini-Mental State Examination, 0–30. A bedside cognitive test. |
| `cdr` | **Clinical Dementia Rating**: 0 = none, 0.5 = very mild, 1 = mild, 2 = moderate. This is the clinical label. |
| `etiv_mm3` | *Estimated total intracranial volume* — the size of the skull cavity. A proxy for **head size**, which does not change with disease. It is here so that brain volumes can be compared between a large man and a small woman. |
| `nwbv` | ***Normalised whole-brain volume*** — the fraction of the skull cavity still filled with brain tissue. Typically 0.84 in a young adult, falling to below 0.70 with atrophy. **This is the atrophy measure. It is the most important column in the table.** |
| `asf` | Atlas scaling factor, the number used to normalise this brain to a template. Essentially 1/eTIV. |


### 1.3 The atrophy signal

**Predict before you run:** the brain shrinks with normal ageing *and* with dementia. Will the two groups' `nwbv` distributions separate cleanly, or overlap heavily?


In [ ]:
plots.plot_class_balance(df['group'], title='Diagnostic groups across all 373 sessions')
plt.show()

plots.plot_by_group(df, 'nwbv', 'group',
                    title='Normalised whole-brain volume — the fraction of skull still filled with brain')
plt.show()

plots.plot_scatter(df['age'], df['nwbv'], colour_by=df['group'],
                   xlabel='age (years)', ylabel='normalised whole-brain volume',
                   title='Brain volume falls with age in everyone. Dementia shifts the whole line down.',
                   legend_title='group')
plt.show()


🧠 **Think first:** The two clouds in the scatter plot overlap a lot. Does that mean brain volume is useless?

<details>
<summary>Click for one good answer</summary>

No — it means it is a *risk marker*, not a *test*. A 78-year-old with an nWBV of 0.68 is much more likely to be demented than one at 0.78, but plenty of individuals sit in the overlap. This is the normal situation for almost every biomarker in medicine, and it is why we measure AUROC (how well the model *ranks* people) rather than demanding a clean separation.

</details>


### 1.4 ✏️ Your turn — the repeated-visit structure

Change `SUBJECT` to look at one person's whole scanning history. Try a few. Some people are stable for years; some visibly decline; the `Converted` group changes diagnosis mid-study.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Pick any subject id from the list printed underneath.
#   Try one 'Converted' subject and one 'Nondemented' subject.
#   Ask: could a model tell these two visits apart, or are they the same person twice?
# ==========================================================================
SUBJECT = 'OAS2_0002'

person = df[df.subject_id == SUBJECT]
display(person[['session_id', 'visit', 'days_since_first_visit', 'age', 'group', 'mmse', 'cdr', 'nwbv']])

visits_each = df.groupby('subject_id').size()
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(visits_each, bins=range(1, visits_each.max() + 2), color='#2c6fbb', align='left', rwidth=0.8)
ax.set_xlabel('number of scans this person contributed'); ax.set_ylabel('number of people')
ax.set_title(f'{len(df)} rows, but only {df.subject_id.nunique()} people')
plt.tight_layout(); plt.show()

converted = df[df.group == 'Converted'].subject_id.unique()
print('Some subjects who converted during the study:', ', '.join(converted[:6]))


### 1.5 The pictures

Now the images. Each row of the table has a matching 64×64 slice, and we will feed those raw pixels to a model in section 3.

**Read this before you look at them.** These slices are *simulated*. We could not ship real OASIS images — they require a signed agreement — so each one is drawn to match that visit's real measured `nwbv` and `etiv_mm3`: less brain tissue means bigger dark ventricles in the middle and wider dark grooves at the surface, which is what atrophy looks like on a real scan. Any model that works on these is really working on the real measurement, laundered through a picture. That is a fair demonstration of the *method* and not evidence about real radiology.


In [ ]:
extra = load_extra('A')
slices, groups_per_slice, nwbv_per_slice = extra['images'], extra['group'], extra['nwbv']
print('Image array shape:', slices.shape, '-> 373 scans, each 64 by 64 pixels')
print('Pixel values run from', slices.min(), 'to', slices.max(), '(0 = black, 1 = white)\n')

# Show the six least atrophied and six most atrophied brains in the study.
order = np.argsort(nwbv_per_slice)
chosen = np.concatenate([order[-6:], order[:6]])
titles = [f'{groups_per_slice[i]}\nnWBV {nwbv_per_slice[i]:.3f}' for i in chosen]
plots.plot_image_grid(slices[chosen], titles=titles, columns=6,
                      title='Top row: most brain tissue.  Bottom row: least (the dark centres are enlarged ventricles).')
plt.show()


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 1.4, find a subject whose `cdr` increases between visits and read their `nwbv` over the same period.
- 🔵 **If you want to write code:** compute each subject's *change* in nWBV per year (`nwbv` difference divided by `days_since_first_visit`/365) and plot it by group. Is the rate of shrinkage more informative than the single measurement?
- ⚫ **Take home:** real longitudinal imaging has a nasty trap: the scanner is recalibrated, or the software is upgraded, between a person's visits, and the apparent 'atrophy' is a software version change. Look up 'longitudinal registration bias' in FreeSurfer.


---
# 2 · Quality control

This module has one flaw so important it deserves the whole section, plus two smaller ones.


### 2.1 Missing values

`ses` and `mmse` have gaps. Before deciding what to do about them, ask *who* is missing.


In [ ]:
plots.plot_missingness(df, title='Missing values across the 373 sessions')
plt.show()

print(df.groupby('group')[['ses', 'mmse', 'cdr']].apply(lambda block: block.isna().mean().round(3)))
print('\n(Values are the fraction missing in each group.)')


### 2.2 The big one — the same person on both sides of the split

Machine learning assumes your test set is made of **people the model has never seen**. Here, 150 people produced 373 scans. If you split the *rows* at random, Mrs Smith's visit 1 lands in training and her visit 2 lands in testing. The model does not need to learn about dementia; it can recognise *her*, and her diagnosis rarely changes between visits.

This is called **subject-level leakage**, and it is probably the single most common fatal flaw in published medical imaging AI. It is easy to introduce and invisible in the results — unless you go looking, which we now do.

The fix is one argument: `split_data(X, y, groups=df['subject_id'])`.


In [ ]:
# Keep the two unambiguous groups so the comparison is clean.
clean = df[df.group.isin(['Nondemented', 'Demented'])].copy()
y_all = (clean.group == 'Demented').astype(int)
volumetric = ['age', 'sex', 'education_years', 'ses', 'nwbv', 'etiv_mm3', 'asf']
X_all = clean[volumetric]

results = {}
for label, grouping in [('split by ROW\n(leaky)', None), ('split by SUBJECT\n(honest)', clean['subject_id'])]:
    X_tr, X_te, y_tr, y_te = split_data(X_all, y_all, groups=grouping)
    shared = set(clean.loc[X_tr.index, 'subject_id']) & set(clean.loc[X_te.index, 'subject_id'])
    model = train_model('random_forest', X_tr, y_tr)
    results[label] = evaluate(model, X_te, y_te)['auroc']
    print(f'{label.splitlines()[0]:<16s} {len(shared):>3d} people appear in BOTH train and test')

plots.plot_score_comparison(list(results), list(results.values()),
                            colours=['#c0392b', '#2c6fbb'], reference=0.5,
                            title='Identical data, identical model. The only difference is how we split.',
                            ylabel='AUROC')
plt.show()
gap = results['split by ROW\n(leaky)'] - results['split by SUBJECT\n(honest)']
print(f'The leaky split inflates AUROC by {gap:+.3f}. Published as-is, that is a fabricated result.')


🧠 **Think first:** Module H (drug discovery) has exactly the same problem with a completely different name. What is it?

<details>
<summary>Click for one good answer</summary>

**Scaffold leakage.** Chemists make dozens of near-identical analogues of one promising molecule. Split those at random and the model sees compound 17a in training and 17b in testing — two atoms different — so it looks brilliant and then fails on a genuinely new scaffold. Same maths, same fix, same `groups=` argument. If you take H as your second module you will see the identical bar chart with chemistry on the axis.

</details>


### 2.3 ✏️ Your turn — feel the leak

Turn grouping on and off yourself, and change the model. The leak's size depends on how good the model is at memorising: flexible models leak more.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Set GROUP_BY_SUBJECT to False, run, then set it back to True.
#   Then repeat with MODEL = 'knn' (a model that literally memorises)
#   and MODEL = 'logistic' (a model that cannot memorise much).
#   Which model is helped most by the cheating?
# ==========================================================================
GROUP_BY_SUBJECT = True
MODEL = 'random_forest'

grouping = clean['subject_id'] if GROUP_BY_SUBJECT else None
X_tr, X_te, y_tr, y_te = split_data(X_all, y_all, groups=grouping)
metrics = evaluate(train_model(MODEL, X_tr, y_tr), X_te, y_te)

plots.plot_score_comparison(list(metrics), list(metrics.values()), reference=0.5,
                            colours=['#2c6fbb' if GROUP_BY_SUBJECT else '#c0392b'] * 5,
                            title=f"{MODEL}, grouped={GROUP_BY_SUBJECT}", ylabel='score')
plt.show()
overlap = len(set(clean.loc[X_tr.index, 'subject_id']) & set(clean.loc[X_te.index, 'subject_id']))
print(f'{overlap} people are in both halves. Anything above zero means the score is not what it claims.')


### 2.4 The other leak — the cognitive test

`mmse` is a cognitive examination. `cdr` — and therefore `group` — was assigned by a clinician who had the cognitive picture in front of them. Predicting `group` from `mmse` is close to predicting the label from the label. Same story as module C, different modality; it is worth seeing how large the effect is here.


In [ ]:
for label, columns in [('imaging + demographics', volumetric),
                       ('imaging only', ['nwbv', 'etiv_mm3', 'asf']),
                       ('+ MMSE (circular)', volumetric + ['mmse'])]:
    X_tr, X_te, y_tr, y_te = split_data(clean[columns], y_all, groups=clean['subject_id'])
    score = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']
    results[label] = score

picked = ['imaging only', 'imaging + demographics', '+ MMSE (circular)']
plots.plot_score_comparison(picked, [results[name] for name in picked],
                            colours=['#2c6fbb', '#2c6fbb', '#e08214'], reference=0.5,
                            title='All subject-grouped. The orange bar is not an imaging result.',
                            ylabel='AUROC')
plt.show()


🧠 **Think first:** Activity cliffs mean near-identical molecules can differ a thousandfold in potency. Which of today's models handles that worst?

<details>
<summary>Click for one good answer</summary>

`knn` — nearest neighbours — is destroyed by it, because its entire assumption is *"things that look alike behave alike"*, and an activity cliff is a counterexample by definition. Tree-based models cope a little better, since they can carve out a sharp region of descriptor space, but our eleven whole-molecule descriptors cannot even *see* the single-atom change that causes the cliff: two compounds either side of one have almost identical molecular weight, logP and ring count.

This is why production models use fingerprints or graph networks, which encode *which* atoms sit where rather than counting them. And it is why chemists remain sceptical of models that report a good average score: the average is dominated by easy compounds, while the interesting ones live on the cliffs.

</details>


### 2.5 QC verdict

**Usable, with one non-negotiable rule and two caveats.**

1. **Always pass `groups=subject_id`.** Everything below does. Without it, every number in this notebook is fiction.
2. `ses` is missing for a substantial minority, and probably not at random — socioeconomic status is harder to collect from exactly the people whose outcomes differ most. We impute it inside the pipeline and keep the caveat.
3. We exclude `mmse` from the imaging models, because this module asks what an *image* can do.

*(**Express path:** you can start from section 3 — run its catch-up cell first and everything below stands alone.)*


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 2.3, set `MODEL = 'knn'` with grouping off and then on, and note which model the leak flatters most.
- 🔵 **If you want to write code:** the `Converted` subjects were dropped in 2.2. Add them back, labelled by their *final* diagnosis, and see what happens. Is that a fair label for their first scan?
- ⚫ **Take home:** read about **immortal time bias** — a related trap where the way you define follow-up guarantees a result. Module E's take-home covers it.


---
# 3 · Build models

Three rungs, in this order:

1. **The numbers a pipeline already extracted** (`nwbv` and friends). Cheap, interpretable, and the bar everything else has to clear.
2. **A support vector machine on the raw image pixels.** No feature extraction at all — 4096 numbers per brain, straight in.
3. **A convolutional neural network on the same pixels.** The method that made medical imaging AI famous.

Every rung uses the same subject-grouped split, so the comparison is fair.


### 🚏 Taking the Express path? Run this one cell first

It rebuilds everything sections 3 and 4 need, so you can start here without having run sections 1 and 2 yourself. **If you did run them, run this anyway** — it just redefines the same things and costs a second.


In [ ]:
# Express catch-up: safe to run whether or not you did sections 1 and 2.
df = load_data('A')
extra = load_extra('A')
slices, groups_per_slice, nwbv_per_slice = extra['images'], extra['group'], extra['nwbv']

# The two unambiguous groups, the target, and the pipeline-extracted measurements.
clean = df[df.group.isin(['Nondemented', 'Demented'])].copy()
y_all = (clean.group == 'Demented').astype(int)
volumetric = ['age', 'sex', 'education_years', 'ses', 'nwbv', 'etiv_mm3', 'asf']
X_all = clean[volumetric]

print(f'{len(clean)} scans from {clean.subject_id.nunique()} people; {y_all.sum()} of them demented.')
print('Ready for section 3.')


### 3.1 Rung one — the extracted measurements

Beat the dumb baseline first.


In [ ]:
X_train, X_test, y_train, y_test = split_data(X_all, y_all, groups=clean['subject_id'])
train_subjects = clean.loc[X_train.index, 'subject_id']
print(f'{len(X_train)} scans from {train_subjects.nunique()} people to train on;')
print(f'{len(X_test)} scans from {clean.loc[X_test.index, "subject_id"].nunique()} DIFFERENT people to test on.\n')

ladder = ['baseline', 'logistic', 'knn', 'random_forest', 'svm']
table = compare_models(ladder, X_train, y_train, X_test, y_test)
display(table)
plots.plot_model_comparison(table, metric='auroc',
                            title='Rung 1: models on the pipeline-extracted volumes (AUROC)')
plt.show()
volumetric_auroc = table.loc['logistic', 'auroc']


### 3.2 ✏️ Your turn — which measurement carries the signal?

Strip the feature set down and see what survives. `nwbv` is doing most of the work; find out how much.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Comment lines in or out (put a # at the start to remove one).
#   Suggested experiments:
#     (a) nwbv alone           -> how far does one number get you?
#     (b) age alone            -> how much is just ageing?
#     (c) everything except nwbv -> is there anything left without the atrophy measure?
# ==========================================================================
FEATURES = [
    'nwbv',              # brain tissue remaining
    'age',
    'sex',
    'education_years',
    'ses',
    'etiv_mm3',          # head size
    'asf',
]

X_tr, X_te, y_tr, y_te = split_data(clean[FEATURES], y_all, groups=clean['subject_id'])
chosen_metrics = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)

plots.plot_score_comparison(list(chosen_metrics), list(chosen_metrics.values()), reference=0.5,
                            colours=['#2c6fbb'] * 5,
                            title=f'{len(FEATURES)} feature(s): ' + ', '.join(FEATURES), ylabel='score')
plt.show()

readable = train_model('logistic', X_tr, y_tr)
names = readable.named_steps['preprocess'].get_feature_names_out()
weights = readable.named_steps['model'].coef_[0]
plots.plot_importance([n.split('__')[-1] for n in names], weights,
                      title='Logistic regression weights (blue pushes towards "demented")',
                      xlabel='coefficient (standardised units)')
plt.show()


### 3.3 Rung two — a support vector machine, straight on the pixels

Now we throw away the extracted numbers and hand the model the picture.

Each 64×64 slice becomes a row of **4096 numbers**, one per pixel. A **support vector machine (SVM)** looks for the boundary that separates the two classes with the widest possible empty corridor around it, and its *kernel* controls the boundary's shape — `'linear'` for a flat plane, `'rbf'` for a curved one.

**Predict before you run:** 4096 measurements per brain and only ~270 training brains. Is that more information than `nwbv` alone, or less?


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# Line the images up with the table rows we kept, and split by SUBJECT again.
keep = np.isin(extra['session_id'], clean['session_id'].to_numpy())
image_X = slices[keep]
image_y = (extra['group'][keep] == 'Demented').astype(int)
image_subjects = extra['subject_id'][keep]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_index, test_index = next(splitter.split(image_X, image_y, image_subjects))
print(f'{len(train_index)} training images, {len(test_index)} test images, no shared subjects: '
      f'{len(set(image_subjects[train_index]) & set(image_subjects[test_index])) == 0}')

flat_train = image_tools.flatten(image_X[train_index])
flat_test = image_tools.flatten(image_X[test_index])
print(f'Each brain is now a row of {flat_train.shape[1]} pixel values.\n')

svm_pixels = Pipeline([
    ('scale', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, random_state=42)),
]).fit(flat_train, image_y[train_index])

svm_probability = svm_pixels.predict_proba(flat_test)[:, 1]
svm_auroc = roc_auc_score(image_y[test_index], svm_probability)
print(f'SVM on raw pixels — held-out AUROC: {svm_auroc:.3f}')

plots.plot_roc_pr(image_y[test_index], svm_probability, title='SVM on 4096 raw pixels per brain')
plt.show()


### 3.4 Compressing the images — eigenbrains

4096 pixels is far more numbers than we have brains, and most of them say the same thing as their neighbours. **Principal component analysis (PCA)** finds the handful of *patterns of variation* that account for most of the difference between these images, and describes each brain as a recipe of those patterns. In face recognition these patterns are famously called *eigenfaces*; on brain images, **eigenbrains**, and they are worth looking at in their own right.

**Predict before you run:** compressing 4096 numbers down to 40 throws information away. Does that help the SVM (less noise to overfit) or hurt it (less signal to use)? Both are plausible. Find out — and then look at 3.9 before you conclude anything from a single split.


In [ ]:
from sklearn.decomposition import PCA

svm_pca = Pipeline([
    ('scale', StandardScaler()),
    ('pca', PCA(n_components=40, random_state=42)),
    ('svm', SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, random_state=42)),
]).fit(flat_train, image_y[train_index])

pca_probability = svm_pca.predict_proba(flat_test)[:, 1]
pca_auroc = roc_auc_score(image_y[test_index], pca_probability)
explained = svm_pca.named_steps['pca'].explained_variance_ratio_.sum()
print(f'40 components keep {explained:.0%} of the variation in the images.')
print(f'SVM on 40 eigenbrains — held-out AUROC: {pca_auroc:.3f}  (raw pixels was {svm_auroc:.3f})\n')

# What do the patterns look like?
components = svm_pca.named_steps['pca'].components_[:8].reshape(-1, 64, 64)
plots.plot_image_grid(components, titles=[f'eigenbrain {i + 1}' for i in range(8)], columns=4,
                      title='The main patterns of variation across these brains', cmap='RdBu_r')
plt.show()

# Where does each brain sit in the first two dimensions?
coordinates = svm_pca.named_steps['pca'].transform(
    svm_pca.named_steps['scale'].transform(flat_test))
plots.plot_scatter(coordinates[:, 0], coordinates[:, 1],
                   colour_by=np.where(image_y[test_index] == 1, 'Demented', 'Nondemented'),
                   xlabel='eigenbrain 1', ylabel='eigenbrain 2',
                   title='Held-out brains in eigenbrain space', legend_title='true group')
plt.show()


### 3.5 ✏️ Your turn — the SVM's three dials

An SVM has few settings, and each one does something you can describe in a sentence:

- **`KERNEL`** — the shape of the boundary. `'linear'` is a flat cut; `'rbf'` bends around clusters.
- **`C`** — how badly the SVM wants to classify every training brain correctly. Large `C` = "get them all right" = memorising. Small `C` = "keep the boundary simple".
- **`N_COMPONENTS`** — how many eigenbrains to keep. Fewer means a blunter but steadier description.

The figure sweeps `C` for whatever kernel and component count you set, so you can see the whole overfitting curve rather than one number.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change these three, then re-run.
#   Suggested experiments:
#     (a) KERNEL='linear' vs 'rbf'   -> does a curved boundary help?
#     (b) N_COMPONENTS = 5, 40, 200  -> how much detail is useful?
#     (c) watch the orange (training) line hit 1.0 as C grows: that is memorising.
# ==========================================================================
KERNEL = 'rbf'          # 'linear' or 'rbf'
N_COMPONENTS = 40       # try 5, 40, 200
C_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0]

train_curve, test_curve = [], []
for c_value in C_VALUES:
    pipeline = Pipeline([
        ('scale', StandardScaler()),
        ('pca', PCA(n_components=N_COMPONENTS, random_state=42)),
        ('svm', SVC(kernel=KERNEL, C=c_value, class_weight='balanced', random_state=42)),
    ]).fit(flat_train, image_y[train_index])
    train_curve.append(balanced_accuracy_score(image_y[train_index], pipeline.predict(flat_train)))
    test_curve.append(balanced_accuracy_score(image_y[test_index], pipeline.predict(flat_test)))

plots.plot_parameter_sweep(C_VALUES, train_curve, test_curve, 'C',
                           title=f'SVM ({KERNEL} kernel, {N_COMPONENTS} eigenbrains) on brain images')
plt.show()
print(f'Best held-out C: {C_VALUES[int(np.argmax(test_curve))]}')


### 3.6 Rung three — a convolutional neural network

An SVM on pixels has no idea that pixel 200 sits next to pixel 201. A **convolutional neural network (CNN)** does. It slides small filters across the image looking for local patterns — an edge, a dark blob — then slides more filters across *those* results, building up from edges to shapes to "large ventricle". That built-in assumption, that nearby pixels belong together, is why CNNs took over medical imaging.

Ours is deliberately tiny: three convolution blocks and one dense layer, about **12000 parameters** — and we have roughly **270 training brains**. Notice that ratio. A model with forty times more parameters than examples can memorise its training set completely, which is why we use dropout, class weighting, and only a dozen epochs.

⏱ **This cell takes about 20–40 seconds on a laptop CPU.** If PyTorch is not installed it automatically trains a small dense network instead, and says so.


In [ ]:
from images import SmallCNN

cnn = SmallCNN(epochs=12, learning_rate=3e-3, channels=8, dropout=0.3, verbose=True)
cnn.fit(image_X[train_index], image_y[train_index],
        validation=(image_X[test_index], image_y[test_index]))

print(f'\nBackend: {cnn.backend}')
print(f'Trainable parameters: {cnn.parameter_count():,}')
print(f'Training brains:      {len(train_index)}')
print(f'Parameters per brain: {cnn.parameter_count() / len(train_index):.0f}')

cnn_probability = cnn.predict_proba(image_X[test_index])[:, 1]
cnn_auroc = roc_auc_score(image_y[test_index], cnn_probability)
print(f'\nCNN — held-out AUROC: {cnn_auroc:.3f}')

# The training curve is the diagnostic that matters.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cnn.history['epoch'], cnn.history['train_loss'], 'o-', color='#e08214', label='training loss')
if cnn.history['validation_loss']:
    ax.plot(cnn.history['epoch'], cnn.history['validation_loss'], 'o-', color='#2c6fbb',
            label='held-out loss')
ax.set_xlabel('epoch (one pass through the training brains)'); ax.set_ylabel('loss (lower is better)')
ax.set_title('If the blue line turns upward while orange keeps falling, it is memorising')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


### 3.7 ✏️ Your turn — the CNN's dials

> **How practitioners actually approach this.** Beat the simple baseline before adding a single layer. Count parameters against sample size — with a few hundred biomedical images, *smaller is usually better*. Use dropout and early stopping by default. And distinguish two very different failures: if training loss falls but held-out loss rises, the model is **overfitting** (shrink it, regularise it, get more data); if *neither* falls, the model or the learning rate is **wrong** (different architecture, different rate).

⏱ Each run of this cell takes roughly as long as 3.6.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change one at a time and re-run. Suggested experiments:
#     (a) EPOCHS = 40      -> does held-out loss start rising? that is overfitting
#     (b) CHANNELS = 2     -> a much smaller network. does it get worse, or better?
#     (c) DROPOUT = 0.0    -> remove the regulariser and watch the curves separate
#     (d) LEARNING_RATE = 0.05 -> too big; the loss bounces or flatlines
# ==========================================================================
EPOCHS = 12
CHANNELS = 8            # filters in the first convolution block
DROPOUT = 0.3           # fraction of connections randomly ignored each step
LEARNING_RATE = 0.003

tuned = SmallCNN(epochs=EPOCHS, learning_rate=LEARNING_RATE, channels=CHANNELS,
                 dropout=DROPOUT, verbose=False)
tuned.fit(image_X[train_index], image_y[train_index],
          validation=(image_X[test_index], image_y[test_index]))

tuned_probability = tuned.predict_proba(image_X[test_index])[:, 1]
print(f'Parameters: {tuned.parameter_count():,}   '
      f'held-out AUROC: {roc_auc_score(image_y[test_index], tuned_probability):.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tuned.history['epoch'], tuned.history['train_loss'], 'o-', color='#e08214', label='training loss')
if tuned.history['validation_loss']:
    ax.plot(tuned.history['epoch'], tuned.history['validation_loss'], 'o-', color='#2c6fbb', label='held-out loss')
    best_epoch = int(np.argmin(tuned.history['validation_loss'])) + 1
    ax.axvline(best_epoch, color='#8a8a8a', linestyle=':')
    ax.annotate(f'early stopping\nwould stop here (epoch {best_epoch})', (best_epoch, ax.get_ylim()[1] * 0.9),
                fontsize=8, color='#8a8a8a', textcoords='offset points', xytext=(6, -10))
ax.set_xlabel('epoch'); ax.set_ylabel('loss')
ax.set_title(f'{CHANNELS} channels, dropout {DROPOUT}, lr {LEARNING_RATE}')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


### 3.8 🔵 Your turn to write code — data augmentation

With 270 training brains, the standard trick is to manufacture more by transforming the ones you have. But **which transformations are anatomically legitimate?**

- Mirroring left↔right is *arguable*: the hemispheres are roughly symmetric, though not identically affected in every dementia.
- Flipping top↔bottom is *nonsense*: it would put the cerebellum above the cortex. A model that learns to cope with upside-down brains has wasted its capacity on an impossible case.

`image_tools.augment_flips` does the legitimate one. Use it.


In [ ]:
# ✅ Worked solution.
augmented_images, augmented_labels = image_tools.augment_flips(
    image_X[train_index], image_y[train_index])
print(f'{len(train_index)} training brains became {len(augmented_images)}.')

augmented_cnn = SmallCNN(epochs=12, verbose=False).fit(
    augmented_images, augmented_labels,
    validation=(image_X[test_index], image_y[test_index]))
augmented_auroc = roc_auc_score(
    image_y[test_index], augmented_cnn.predict_proba(image_X[test_index])[:, 1])

plots.plot_score_comparison(['original', 'left-right augmented'], [cnn_auroc, augmented_auroc],
                            reference=0.5, title='Does doubling the training set help?', ylabel='AUROC')
plt.show()

# Why this often helps only a little here: a left-right mirror of a roughly symmetric phantom
# is nearly the same picture, so it adds little genuinely new information. Augmentation buys
# the most when the transformation reflects a variation the model will really meet — different
# head positioning, different scanner intensity scaling, slightly different slice level. Those
# are the augmentations a neuroimager would reach for, and all of them are defensible in a way
# that a vertical flip is not.


### 3.9 All four approaches — with honest error bars

You now have four held-out AUROCs from **one** split of 34 test people. Before comparing them, ask how much a single split can be trusted at that sample size.

So we repeat the whole thing across several different subject-grouped splits and plot the spread. This is the most important figure in the module, and it usually surprises people.

⏱ **This cell takes 1–2 minutes** — it retrains everything several times. Start it and read the text below while it runs.


In [ ]:
N_REPEATS = 5   # different random subject-grouped splits

collected = {'extracted volumes (logistic)': [], 'raw pixels (SVM)': [],
             'eigenbrains (PCA + SVM)': [], 'raw pixels (CNN)': []}

for seed in range(N_REPEATS):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr, te = next(splitter.split(image_X, image_y, image_subjects))
    flat_tr, flat_te = image_tools.flatten(image_X[tr]), image_tools.flatten(image_X[te])

    # rung 1: the pipeline-extracted volumes, on the SAME sessions
    sessions = extra['session_id'][keep]
    table_train = clean.set_index('session_id').loc[sessions[tr]]
    table_test = clean.set_index('session_id').loc[sessions[te]]
    volumes = train_model('logistic', table_train[volumetric], pd.Series(image_y[tr], index=table_train.index))
    collected['extracted volumes (logistic)'].append(
        roc_auc_score(image_y[te], volumes.predict_proba(table_test[volumetric])[:, 1]))

    # rung 2a: SVM on raw pixels
    raw = Pipeline([('scale', StandardScaler()),
                    ('svm', SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True,
                                random_state=42))]).fit(flat_tr, image_y[tr])
    collected['raw pixels (SVM)'].append(roc_auc_score(image_y[te], raw.predict_proba(flat_te)[:, 1]))

    # rung 2b: SVM on eigenbrains
    compressed = Pipeline([('scale', StandardScaler()),
                           ('pca', PCA(n_components=40, random_state=42)),
                           ('svm', SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True,
                                       random_state=42))]).fit(flat_tr, image_y[tr])
    collected['eigenbrains (PCA + SVM)'].append(
        roc_auc_score(image_y[te], compressed.predict_proba(flat_te)[:, 1]))

    # rung 3: the CNN
    network = SmallCNN(epochs=12, verbose=False, seed=seed).fit(image_X[tr], image_y[tr])
    collected['raw pixels (CNN)'].append(roc_auc_score(image_y[te], network.predict_proba(image_X[te])[:, 1]))
    print(f'  split {seed + 1}/{N_REPEATS} done')

fig, ax = plt.subplots(figsize=(8, 4.2))
for position, (name, scores) in enumerate(collected.items()):
    ax.scatter(scores, [position] * len(scores), s=55, color='#2c6fbb', zorder=3, alpha=0.8)
    ax.plot([np.mean(scores)] * 2, [position - 0.22, position + 0.22], color='#e08214', linewidth=3)
    ax.annotate(f'{np.mean(scores):.3f} ± {np.std(scores):.3f}', (max(scores), position),
                textcoords='offset points', xytext=(12, -4), fontsize=9)
ax.axvline(0.5, color='#8a8a8a', linestyle='--')
ax.set_yticks(range(len(collected)), list(collected), fontsize=10)
ax.set_xlim(0.35, 1.05)
ax.set_xlabel('held-out AUROC (each dot is one subject-grouped split)')
ax.set_title(f'Four approaches, {N_REPEATS} splits each. Do the clouds actually separate?')
plt.tight_layout(); plt.show()

for name, scores in collected.items():
    print(f'  {name:<32s} {np.mean(scores):.3f} ± {np.std(scores):.3f}   '
          f'(worst {min(scores):.3f}, best {max(scores):.3f})')


🧠 **Think first:** Look at the spread. Can you honestly say the CNN — the most sophisticated method here — beats a single number called `nwbv`?

<details>
<summary>Click for one good answer</summary>

Almost certainly not, and that is the most useful thing in this module. With 34 held-out people, one split's AUROC bounces around by 0.1 or more, which is larger than any gap between the four methods. **A difference smaller than the spread is not a difference.**

Three reasons the fancy method does not pull ahead here:

1. **Sample size.** Deep learning's advantage appears with tens of thousands of examples. With 250 brains, a 12000-parameter network spends most of its capacity memorising.
2. **The pipeline already did the hard part.** `nwbv` is the output of decades of neuroimaging research into how to summarise a brain. Beating a great hand-crafted feature from scratch is a high bar.
3. **Our slices are simple.** They were drawn from `nwbv`, so there is genuinely no extra information in the pixels to find. On real scans there *is* more — regional atrophy patterns that distinguish Alzheimer's from frontotemporal dementia, for instance — and that is where CNNs earn their keep.

The transferable lesson: **"we used deep learning" is not a result**, and neither is a single held-out number. The result is whether it beat the simple thing, repeatedly, on an honest split.

</details>


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 3.7, set `EPOCHS = 40` and find the epoch where held-out loss stops improving. That is where early stopping would have stopped.
- 🔵 **If you want to write code:** complete the augmentation TODO in 3.8, then try adding small random shifts (`np.roll`) as a second augmentation.
- ⚫ **Take home:** swap the CNN for a **frozen pretrained backbone**: load a network trained on natural images, use it as a fixed feature extractor, and fit a logistic regression on its output. This is what people actually do when they have a few hundred medical images.


---
# 4 · Read the results

One imaging model gets the full treatment. Rather than pick a favourite in advance, the next cell takes whichever of the three image approaches scored best on *this* split — and section 3.9 has already warned you how much that choice depends on the split.


### 4.1 The standard views


In [ ]:
candidates = {'SVM on raw pixels': (svm_probability, svm_auroc),
              'SVM on eigenbrains': (pca_probability, pca_auroc),
              'small CNN': (cnn_probability, cnn_auroc)}
headline_name = max(candidates, key=lambda name: candidates[name][1])
headline_probability, headline_auroc = candidates[headline_name]
print(f'Best image model on this split: {headline_name} (AUROC {headline_auroc:.3f}).')
print('Section 3.9 showed how much that ranking moves between splits — keep it in mind.\n')

y_true = image_y[test_index]
predicted = (headline_probability >= 0.5).astype(int)

plots.plot_confusion(y_true, predicted, labels=('nondemented', 'demented'),
                     title=f'Held-out brains: {headline_name}')
plt.show()

plots.plot_roc_pr(y_true, headline_probability, title=f'{headline_name}, held-out subjects')
plt.show()

plots.plot_calibration(y_true, headline_probability)
plt.show()


**In this setting a false negative is a person told their scan looks fine when it does not.** They leave without a referral, without a conversation about what is happening, and without the chance to plan while they still can. A false positive is a healthy person put through months of anxiety and a battery of further tests. Whether you would rather make more of one or the other is a clinical and ethical judgement, and it is made by choosing the threshold — not by the algorithm.


### 4.2 ✏️ Your turn — look at the brains it got wrong

This is the payoff of working with images: you can *look* at the failures.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try 'wrong', then 'right', then 'uncertain'.
#   Ask yourself: do the mistakes look different from the successes,
#   or does the model just fail on brains that genuinely look borderline?
# ==========================================================================
SHOW = 'wrong'          # 'wrong', 'right' or 'uncertain'
DECISION_THRESHOLD = 0.5

at_threshold = (headline_probability >= DECISION_THRESHOLD).astype(int)
if SHOW == 'wrong':
    which = np.where(at_threshold != y_true)[0]
elif SHOW == 'right':
    which = np.where(at_threshold == y_true)[0]
else:
    which = np.argsort(np.abs(headline_probability - 0.5))
which = which[:12]

names = ['nondemented', 'demented']
captions = [f'true: {names[y_true[i]]}\nsaid: {headline_probability[i]:.2f}' for i in which]
plots.plot_image_grid(image_X[test_index][which], titles=captions, columns=6,
                      title=f'Held-out brains the model got {SHOW} (threshold {DECISION_THRESHOLD})')
plt.show()

plots.plot_threshold_sweep(y_true, headline_probability, chosen=DECISION_THRESHOLD)
plt.show()
print(f'{len(np.where(at_threshold != y_true)[0])} of {len(y_true)} held-out scans misclassified '
      f'at threshold {DECISION_THRESHOLD}.')


### 4.3 Which brains does it get wrong — and who are they?

Errors are never evenly spread. Break them down by age and sex.


In [ ]:
error_table = pd.DataFrame({
    'correct': (predicted == y_true).astype(int),
    'age': extra['age'][keep][test_index],
    'sex': extra['sex'][keep][test_index],
    'nwbv': extra['nwbv'][keep][test_index],
})
error_table['age_band'] = pd.cut(error_table['age'], [55, 70, 78, 85, 100],
                                 labels=['<70', '70-78', '78-85', '85+'])

for subgroup in ['sex', 'age_band']:
    plots.plot_subgroup_errors(error_table, subgroup, 'correct',
                               title=f'Proportion of held-out scans classified correctly, by {subgroup}')
    plt.show()

plots.plot_scatter(error_table['nwbv'], headline_probability, colour_by=np.where(y_true == 1, 'demented', 'nondemented'),
                   xlabel='normalised whole-brain volume (the real measurement)',
                   ylabel='probability of dementia the model assigned',
                   title='The model reading the picture vs the number the picture was drawn from',
                   legend_title='true group')
plt.show()


### 4.4 Your headline result

For your own notes. **No leaderboard, no comparison with anyone else's module** — the point of the wrap-up session is that these numbers are not comparable across modalities anyway.


In [ ]:
from sklearn.metrics import balanced_accuracy_score, average_precision_score
from sklearn.metrics import confusion_matrix as sk_confusion

tn, fp, fn, tp = sk_confusion(y_true, predicted, labels=[0, 1]).ravel()
headline = {
    'balanced_accuracy': balanced_accuracy_score(y_true, predicted),
    'auroc': headline_auroc,
    'auprc': average_precision_score(y_true, headline_probability),
    'sensitivity': tp / max(tp + fn, 1),
    'specificity': tn / max(tn + fp, 1),
}
plots.plot_score_comparison(list(headline), list(headline.values()), reference=0.5,
                            colours=['#2c6fbb'] * 5,
                            title=f'Module A — {headline_name}, subject-grouped held-out scans',
                            ylabel='score')
plt.show()
print(f'Trained on {len(train_index)} scans from {len(set(image_subjects[train_index]))} people.')
print(f'Tested on {len(test_index)} scans from {len(set(image_subjects[test_index]))} DIFFERENT people.')
for name, value in headline.items():
    print(f'  {name:<20s} {value:.3f}')


### 4.5 What would have to be true before this touched a patient?

1. **Real images.** Our slices encode one summary number. A real scan carries regional detail that distinguishes Alzheimer's from vascular dementia, from frontotemporal dementia, from normal pressure hydrocephalus — the differential diagnosis that actually matters in a memory clinic.
2. **More than one scanner.** A CNN will happily learn "this hospital's scanner" instead of "this patient's brain". Multi-site validation is mandatory.
3. **The label is a clinical judgement.** CDR is assigned by a person, from an interview. Two clinicians disagree more often than you would like. No model beats its labels.
4. **150 people is a pilot, not evidence.**
5. **Coverage.** OASIS-2 is a single American city's volunteers, mostly white, mostly educated, all willing to be scanned repeatedly. Brain volume norms genuinely differ by ancestry and by head size, and a model tuned on this cohort will be least reliable for the people least represented in it.

---

### 🧠 Final question for the group discussion

MRI is expensive, requires a scanner and a radiographer, and takes half an hour. A blood test (module C) costs a few euros. Given what you have seen — **where in the diagnostic pathway does imaging belong?** Screening everybody, or confirming a suspicion someone else raised?


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 4.2 set `DECISION_THRESHOLD = 0.3` and look at how the misclassified brains change.
- 🔵 **If you want to write code:** retrain the eigenbrain SVM using only subjects' **first** visits, so every person contributes exactly one row. Does the honest score change? What did you give up?
- ⚫ **Take home:** the real version of this module uses the differential-diagnosis framing: not "dementia yes/no" but "which dementia". Look up how AD, FTD and DLB differ in their atrophy patterns, and what that means for a model that only sees one slice.
